# Substation Shifting Proportionality Test

**Question:** Is the per-household shift rate the same across all substations, or do some substations differ beyond random noise?

---

## Why not Pearson correlation on totals?

Correlating *total shift* with *n_households* (r ≈ 0.995) is nearly tautological:
`total_shift = per_capita_rate × n_households`, so the 15:1 range in population size (297–4,536 HHs) dominates the correlation completely. This confirms that bigger substations shift more aggregate load — which is obvious — not that the *rate* is proportional.

## Why not chi-square?

Chi-square tests independence between categorical variables. Both shifting and population are continuous ratio-scale quantities; binning them is arbitrary and lossy.

## Correct approach: household-level one-way ANOVA

We use each individual household's per-day shift as the unit of observation (n ≈ 28,195), with **substation as the grouping factor (21 levels)**:

- **H₀:** All substations have the same mean per-household shift rate — population size is the only predictor.
- **H₁:** At least one substation has a systematically different per-household rate.
- **F-ratio** = between-substation variance / within-substation variance.
- **η²** (eta-squared) = SS_between / SS_total — fraction of shift variance explained by substation membership.

This is valid because we have true replication: hundreds to thousands of households per substation. ANOVA on the *substation* level (n = 1 per group) would be meaningless — ANOVA on the *household* level is exactly right.

---

**Red day only:** 2014-07-03 (only red day in the dataset; strongest Tempo price signal).  
**TOU tariff window — hours 13–19:** Hours where TOU strictly suppresses load (verified by inspection: 0 % of households show increases in h13–19, 8.9 % show increases only in h20–24 = reallocated demand).

In [ ]:
import os
import zipfile
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

BASE_DIR    = '/home/user/capstone_visuals'
ZIP_PATH    = os.path.join(BASE_DIR, 'dist_net.zip')
CONTROL_CSV = os.path.join(BASE_DIR, 'data', 'control_profile.csv')
TEMPO_CSV   = os.path.join(BASE_DIR, 'data', 'tempo_shifted_profile.csv')
TOU_ZIP     = os.path.join(BASE_DIR, 'data', 'tou_shifted_profile.zip')
EXTRACT_DIR = '/tmp/dist_net'
OUTPUT_DIR  = os.path.join(BASE_DIR, 'output')

RED_DAY       = '2014-07-03'
HOUR_COLS     = [str(i) for i in range(1, 25)]   # all 24 hours — Tempo
TOU_HOUR_COLS = [str(i) for i in range(13, 20)]  # TOU tariff window: hours 13-19

def normalize_hid(x):
    try:    return str(int(float(x)))
    except: return None

print('Imports OK')
print('Red day:      ', RED_DAY)
print('Tempo hours:  ', f'{HOUR_COLS[0]}-{HOUR_COLS[-1]}  ({len(HOUR_COLS)} hours)')
print('TOU window:   ', f'{TOU_HOUR_COLS[0]}-{TOU_HOUR_COLS[-1]}  ({len(TOU_HOUR_COLS)} hours)')

## 1. Build HID → Substation Mapping from Shapefiles

In [ ]:
# Extract network zip if needed
flag = os.path.join(EXTRACT_DIR, '.done')
if not os.path.exists(flag):
    print('Extracting dist_net.zip...')
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(EXTRACT_DIR)
    open(flag, 'w').close()

# Load all H-nodes across substations to build HID → substation map
content = os.path.join(EXTRACT_DIR, 'content', 'output')
all_nodes = []

for folder in sorted(os.listdir(content)):
    fp = os.path.join(content, folder, f'{folder}-nodelist-HID.shp')
    if not os.path.exists(fp):
        continue
    gdf = gpd.read_file(fp)
    gdf['substation'] = folder
    all_nodes.append(gdf)

nodes = pd.concat(all_nodes, ignore_index=True)
h_nodes = nodes[nodes['label'] == 'H'].copy()
h_nodes['hid_key'] = h_nodes['hid'].apply(normalize_hid)
h_nodes = h_nodes.dropna(subset=['hid_key'])

# Household count per substation
hh_counts = h_nodes.groupby('substation')['hid_key'].nunique().rename('n_households')

print(f'Total H-nodes: {len(h_nodes):,}')
print(f'Substations:   {hh_counts.shape[0]}')
print()
print(hh_counts.sort_values(ascending=False).to_string())

## 2. Load CSVs, Match Households, Compute Per-Household Shifts

For each household matched across control ↔ scenario on the red day:
- **Tempo shift** = sum(tempo_h1-24) − sum(ctrl_h1-24)
- **TOU shift** = sum(ctrl_h13-19) − sum(tou_h13-19)  (load shed; always ≥ 0 in tariff window)

In [ ]:
# Load and filter to red day
print('Loading control...')
ctrl = pd.read_csv(CONTROL_CSV)
ctrl['hid_key'] = ctrl['hid'].apply(normalize_hid)
ctrl = ctrl[ctrl['date'] == RED_DAY]

print('Loading tempo...')
tempo = pd.read_csv(TEMPO_CSV)
tempo['hid_key'] = tempo['hid'].apply(normalize_hid)
tempo = tempo[tempo['date'] == RED_DAY]

print('Loading TOU...')
with zipfile.ZipFile(TOU_ZIP) as z:
    with z.open('tou_shifted_profile.csv') as f:
        tou = pd.read_csv(f)
tou['hid_key'] = tou['hid'].apply(normalize_hid)
tou = tou[tou['date'] == RED_DAY]

print(f'\nControl  — red day rows: {len(ctrl):,}   unique HIDs: {ctrl["hid_key"].nunique():,}')
print(f'Tempo    — red day rows: {len(tempo):,}   unique HIDs: {tempo["hid_key"].nunique():,}')
print(f'TOU      — red day rows: {len(tou):,}   unique HIDs: {tou["hid_key"].nunique():,}')

In [ ]:
# Add substation labels
hid_sub = h_nodes[['hid_key', 'substation']].drop_duplicates(subset='hid_key')

def add_substation(df):
    return df.merge(hid_sub, on='hid_key', how='inner')

ctrl_s  = add_substation(ctrl)
tempo_s = add_substation(tempo)
tou_s   = add_substation(tou)

# Build household-level shift table (one row per household)
# Tempo: total 24h shift per household
ctrl_24h  = ctrl_s [['hid_key','substation'] + HOUR_COLS].copy()
tempo_24h = tempo_s[['hid_key'] + HOUR_COLS].copy()
ctrl_24h ['ctrl_24h']  = ctrl_24h [HOUR_COLS].sum(axis=1)
tempo_24h['tempo_24h'] = tempo_24h[HOUR_COLS].sum(axis=1)
hh_tempo = ctrl_24h[['hid_key','substation','ctrl_24h']].merge(
    tempo_24h[['hid_key','tempo_24h']], on='hid_key')
hh_tempo['shift_tempo'] = hh_tempo['tempo_24h'] - hh_tempo['ctrl_24h']

# TOU: tariff-window (h13-19) load shed per household
ctrl_win = ctrl_s [['hid_key','substation'] + TOU_HOUR_COLS].copy()
tou_win  = tou_s  [['hid_key'] + TOU_HOUR_COLS].copy()
ctrl_win ['ctrl_win'] = ctrl_win [TOU_HOUR_COLS].sum(axis=1)
tou_win  ['tou_win']  = tou_win  [TOU_HOUR_COLS].sum(axis=1)
hh_tou = ctrl_win[['hid_key','substation','ctrl_win']].merge(
    tou_win[['hid_key','tou_win']], on='hid_key')
hh_tou['shift_tou'] = hh_tou['ctrl_win'] - hh_tou['tou_win']  # load shed >= 0

print(f'Household-level Tempo rows: {len(hh_tempo):,}')
print(f'Household-level TOU rows:   {len(hh_tou):,}')
print()
print('Households per substation:')
print(hh_tempo.groupby('substation').size().sort_values(ascending=False).to_string())

In [ ]:
# Per-substation summary: mean per-household rate and 95% CI
def substation_summary(hh_df, shift_col):
    grp = hh_df.groupby('substation')[shift_col]
    summary = grp.agg(n='count', mean='mean', std='std').reset_index()
    summary['se']    = summary['std'] / np.sqrt(summary['n'])
    summary['ci95']  = 1.96 * summary['se']
    return summary.set_index('substation').sort_values('n', ascending=False)

sub_tempo = substation_summary(hh_tempo, 'shift_tempo')
sub_tou   = substation_summary(hh_tou,   'shift_tou')

print('=== Per-substation mean shift rate (kWh/household) ===')
print()
print('TEMPO (24h net shift):')
print(sub_tempo[['n','mean','std','se','ci95']].round(5).to_string())
print()
print('TOU (h13-19 load shed):')
print(sub_tou[['n','mean','std','se','ci95']].round(5).to_string())
print()
print(f'Tempo  — overall CV: {100*sub_tempo["mean"].std()/sub_tempo["mean"].abs().mean():.1f}%')
print(f'TOU    — overall CV: {100*sub_tou["mean"].std()/sub_tou["mean"].mean():.1f}%')

## 3. One-Way ANOVA: Does Substation Explain Per-Household Shift Rate?

H₀: All substation group means are equal (substation membership adds no explanatory power beyond noise).  
H₁: At least one substation has a different mean per-household shift rate.

**η²** = SS_between / SS_total — proportion of total variance in household-level shifts explained by substation.  
**ω²** = (SS_between − (k−1)·MS_within) / (SS_total + MS_within) — bias-corrected version of η².

In [ ]:
def one_way_anova(hh_df, shift_col, scenario_label):
    groups = [grp[shift_col].values for _, grp in hh_df.groupby('substation')]
    k  = len(groups)                      # number of substations
    N  = sum(len(g) for g in groups)      # total households
    grand_mean = hh_df[shift_col].mean()

    # Sums of squares
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_within  = sum(((g - g.mean())**2).sum() for g in groups)
    ss_total   = ss_between + ss_within

    df_between = k - 1
    df_within  = N - k
    ms_between = ss_between / df_between
    ms_within  = ss_within  / df_within

    F = ms_between / ms_within
    p = stats.f.sf(F, df_between, df_within)

    # Effect sizes
    eta2  = ss_between / ss_total
    omega2 = (ss_between - df_between * ms_within) / (ss_total + ms_within)

    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '(ns)'
    print(f'=== One-Way ANOVA: {scenario_label} ===')
    print(f'  k = {k} substations   N = {N:,} households')
    print(f'  F({df_between}, {df_within}) = {F:.4f}   p = {p:.2e}  {sig}')
    print(f'  η²  = {eta2:.4f}  ({100*eta2:.2f}% of variance explained by substation)')
    print(f'  ω²  = {omega2:.4f}  (bias-corrected)')
    print()
    return F, p, eta2, omega2

F_t, p_t, eta2_t, omega2_t = one_way_anova(hh_tempo, 'shift_tempo', 'TEMPO (24h net shift, red day)')
F_u, p_u, eta2_u, omega2_u = one_way_anova(hh_tou,   'shift_tou',   'TOU   (h13-19 load shed, red day)')

## 4. Visualisation: Per-Substation Mean Shift Rates with 95% CIs

Substations sorted by household count (largest → smallest). Overlapping CIs suggest rates are statistically indistinguishable; non-overlapping CIs identify substations that genuinely differ.

In [ ]:
BG  = '#06090f';  TXT = '#c8d8e8';  DIM = '#2a3f55'
C_TEMPO = '#ff8c00';  C_TOU = '#00b4d8';  C_MEAN = '#ffffff'

plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG,
    'text.color': TXT, 'axes.labelcolor': TXT,
    'xtick.color': TXT, 'ytick.color': TXT,
    'axes.edgecolor': DIM, 'grid.color': DIM,
    'font.family': 'monospace', 'font.size': 9,
})

fig, axes = plt.subplots(1, 2, figsize=(15, 6), facecolor=BG)

for ax, sub_df, color, scenario, ylabel, F, p, eta2 in [
    (axes[0], sub_tempo, C_TEMPO, 'TEMPO', 'Mean shift per household (kWh, 24h)', F_t, p_t, eta2_t),
    (axes[1], sub_tou,   C_TOU,   'TOU',   'Mean load shed per household (kWh, h13-19)', F_u, p_u, eta2_u),
]:
    subs = sub_df.index.tolist()          # already sorted by n descending
    y    = np.arange(len(subs))
    means  = sub_df['mean'].values
    ci95   = sub_df['ci95'].values
    ns     = sub_df['n'].values
    grand  = np.average(means, weights=ns)

    ax.barh(y, means, color=color, alpha=0.5, height=0.6, zorder=3)
    ax.errorbar(means, y, xerr=ci95, fmt='none', color='white',
                elinewidth=0.9, capsize=3, capthick=0.9, zorder=5)
    ax.axvline(grand, color=C_MEAN, lw=1.2, linestyle='--', alpha=0.8,
               label=f'Weighted mean = {grand:.4f}')
    ax.axvline(0,     color=DIM,    lw=0.7, zorder=2)

    ax.set_yticks(y)
    ax.set_yticklabels([f'{s}  (n={n:,})' for s, n in zip(subs, ns)], fontsize=7)
    ax.set_xlabel(ylabel, fontsize=8)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    ax.set_title(f'{scenario}  |  F={F:.1f}  p{sig}  η²={eta2:.3f}\n'
                 'Error bars = 95% CI on household mean', fontsize=9)
    ax.grid(axis='x', lw=0.35, alpha=0.5)
    ax.legend(fontsize=7.5, facecolor='#0c1622', edgecolor=DIM, labelcolor=TXT)

fig.suptitle('Per-Household Shift Rates by Substation  |  Red Day (2014-07-03)',
             color='white', fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, '07_shifting_proportionality.png')
fig.savefig(out_path, dpi=150, bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved to', out_path)

## 5. Summary

In [ ]:
rows = []
for scenario, hh_df, shift_col, F, p, eta2, omega2 in [
    ('Tempo (24h net)',     hh_tempo, 'shift_tempo', F_t, p_t, eta2_t, omega2_t),
    ('TOU (h13-19 shed)',   hh_tou,   'shift_tou',   F_u, p_u, eta2_u, omega2_u),
]:
    grand_mean = hh_df[shift_col].mean()
    cv = 100 * hh_df.groupby('substation')[shift_col].mean().std() / abs(grand_mean)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    rows.append({
        'Scenario':       scenario,
        'N (households)': len(hh_df),
        'k (substations)': hh_df['substation'].nunique(),
        'Grand mean (kWh/HH)': round(grand_mean, 5),
        'CV of substation means (%)': round(cv, 1),
        'F-statistic': round(F, 2),
        'p-value': f'{p:.2e}',
        'Significant': sig,
        'eta2': round(eta2, 4),
        'omega2': round(omega2, 4),
    })

summary = pd.DataFrame(rows)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
print(summary.T.to_string())